# Проверка целостности feature-датасетов в S3

Ноутбук проверяет дневные parquet-партиции `unified_dataset` и `hmm_dataset`:

- есть ли все дневные партиции в заданном или фактическом диапазоне дат;
- непрерывен ли минутный `timestamp` внутри партиций и между соседними днями;
- нет ли дубликатов, невалидных timestamp или строк не из своего `date=...` раздела;
- нет ли `NaN`/`None` в середине ряда по каждой переменной;
- нет ли `inf`/`-inf` в числовых колонках.

Для лагов и rolling/window-признаков разрешены пустые значения только до первого валидного значения в колонке. Любой пропуск после первого валидного значения считается разрывом.

In [2]:
from __future__ import annotations

import io
import json
import os
import re
from dataclasses import dataclass, field
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from IPython.display import display

In [3]:
# Основной конфиг проверки.
BUCKET = os.getenv("YC_BUCKET", "binance-data-downloader")
SYMBOL = "ADAUSDT"
INTERVAL = "1m"
EXPECTED_FREQ = "1min"

# Если оставить None, ноутбук возьмет фактический min/max из найденных партиций.
# Укажите строки YYYY-MM-DD, если нужно зафиксировать контрактный диапазон.
START_DATE = None
END_DATE = None

DATASETS = {
    "unified_dataset": {
        "prefix": "features/unified_dataset",
        "symbol": SYMBOL,
        "interval": INTERVAL,
        "expected_rows_per_full_day": 1440,
    },
    "hmm_dataset": {
        "prefix": "features/hmm_dataset",
        "symbol": SYMBOL,
        "interval": INTERVAL,
        "expected_rows_per_full_day": 1440,
    },
}

# Ограничитель числа примеров в отчетах, чтобы ноутбук не раздувался на больших разрывах.
MAX_EXAMPLES_PER_COLUMN = 20
MAX_EXAMPLES_PER_DATASET = 2000

REPORT_DIR = Path("analysis/data_integrity_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
load_dotenv(".env")

required_env = ("YC_ENDPOINT", "YC_REGION", "YC_ACCESS_KEY_ID", "YC_SECRET_ACCESS_KEY")
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    raise RuntimeError(f"Missing S3 environment variables: {', '.join(missing_env)}")

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
)

In [5]:
DATE_RE = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")


def dataset_prefix(config: dict) -> str:
    return (
        f"{config['prefix'].strip('/')}/symbol={config['symbol']}/"
        f"interval={config['interval']}/"
    )


def partition_key(config: dict, date: str) -> str:
    return f"{dataset_prefix(config)}date={date}/data.parquet"


def list_dataset_partitions(bucket: str, config: dict) -> pd.DataFrame:
    prefix = dataset_prefix(config)
    rows = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            match = DATE_RE.search(f"/{key}")
            if not match:
                continue
            rows.append(
                {
                    "date": match.group(1),
                    "key": key,
                    "size_bytes": obj.get("Size", 0),
                    "last_modified": obj.get("LastModified"),
                }
            )
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def read_s3_parquet(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))


def expected_dates(found_dates: list[str]) -> pd.DatetimeIndex:
    if START_DATE is not None:
        start = pd.Timestamp(START_DATE)
    else:
        start = pd.Timestamp(min(found_dates))
    if END_DATE is not None:
        end = pd.Timestamp(END_DATE)
    else:
        end = pd.Timestamp(max(found_dates))
    return pd.date_range(start, end, freq="D")


def compact_timestamp_runs(timestamps: pd.Series | pd.DatetimeIndex, freq: str = EXPECTED_FREQ) -> list[dict]:
    if len(timestamps) == 0:
        return []
    idx = pd.DatetimeIndex(pd.to_datetime(timestamps, utc=True)).sort_values().unique()
    expected = pd.date_range(idx.min(), idx.max(), freq=freq)
    missing = expected.difference(idx)
    if len(missing) == 0:
        return []
    breaks = missing.to_series().diff().ne(pd.Timedelta(freq)).cumsum().to_numpy()
    result = []
    for group_id in np.unique(breaks):
        group = missing[breaks == group_id]
        result.append(
            {
                "start": group[0].isoformat(),
                "end": group[-1].isoformat(),
                "missing_rows": len(group),
            }
        )
    return result


def compact_null_runs(timestamp: pd.Series, is_null: pd.Series, limit: int) -> list[dict]:
    missing_ts = pd.DatetimeIndex(timestamp.loc[is_null].sort_values().unique())
    if len(missing_ts) == 0:
        return []
    breaks = missing_ts.to_series().diff().ne(pd.Timedelta(EXPECTED_FREQ)).cumsum().to_numpy()
    result = []
    for group_id in np.unique(breaks):
        group = missing_ts[breaks == group_id]
        result.append(
            {
                "start": group[0].isoformat(),
                "end": group[-1].isoformat(),
                "rows": len(group),
            }
        )
        if len(result) >= limit:
            break
    return result

In [6]:
@dataclass
class ColumnState:
    seen_valid: bool = False
    first_valid_ts: pd.Timestamp | None = None
    first_gap_ts: pd.Timestamp | None = None
    gap_rows: int = 0
    examples: list[dict] = field(default_factory=list)
    leading_null_rows: int = 0
    total_rows: int = 0
    non_null_rows: int = 0


def update_null_state(states: dict[str, ColumnState], frame: pd.DataFrame, dataset_name: str, date: str) -> list[dict]:
    timestamp = frame["timestamp"]
    issues = []
    for column in frame.columns:
        if column == "timestamp":
            continue
        state = states.setdefault(column, ColumnState())
        values = frame[column]
        is_null = values.isna()
        state.total_rows += len(values)
        state.non_null_rows += int((~is_null).sum())

        if not state.seen_valid:
            valid_positions = np.flatnonzero((~is_null).to_numpy())
            if len(valid_positions) == 0:
                state.leading_null_rows += len(values)
                continue
            first_pos = int(valid_positions[0])
            state.leading_null_rows += int(is_null.iloc[:first_pos].sum())
            state.seen_valid = True
            state.first_valid_ts = timestamp.iloc[first_pos]
            check_mask = is_null.copy()
            check_mask.iloc[: first_pos + 1] = False
        else:
            check_mask = is_null

        if check_mask.any():
            state.gap_rows += int(check_mask.sum())
            if state.first_gap_ts is None:
                state.first_gap_ts = timestamp.loc[check_mask].iloc[0]
            if len(state.examples) < MAX_EXAMPLES_PER_COLUMN:
                remaining = MAX_EXAMPLES_PER_COLUMN - len(state.examples)
                state.examples.extend(compact_null_runs(timestamp, check_mask, remaining))
            issues.append(
                {
                    "dataset": dataset_name,
                    "date": date,
                    "column": column,
                    "gap_rows_in_partition": int(check_mask.sum()),
                    "first_gap_in_partition": timestamp.loc[check_mask].iloc[0].isoformat(),
                }
            )
    return issues


def summarize_null_states(dataset_name: str, states: dict[str, ColumnState]) -> pd.DataFrame:
    rows = []
    for column, state in sorted(states.items()):
        rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "status": "ok" if state.seen_valid and state.gap_rows == 0 else "FAIL",
                "total_rows": state.total_rows,
                "non_null_rows": state.non_null_rows,
                "leading_null_rows_allowed": state.leading_null_rows,
                "gap_rows_after_first_valid": state.gap_rows,
                "first_valid_ts": None if state.first_valid_ts is None else state.first_valid_ts.isoformat(),
                "first_gap_ts": None if state.first_gap_ts is None else state.first_gap_ts.isoformat(),
                "gap_examples": json.dumps(state.examples, ensure_ascii=False),
            }
        )
    return pd.DataFrame(rows)

In [7]:
def check_partition_frame(frame: pd.DataFrame, dataset_name: str, date: str, expected_rows: int | None) -> tuple[dict, list[dict], list[dict]]:
    partition_issues = []
    finite_issues = []

    if "timestamp" not in frame.columns:
        raise ValueError(f"{dataset_name} {date}: missing timestamp column")

    frame = frame.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True, errors="coerce")
    invalid_ts = int(frame["timestamp"].isna().sum())
    if invalid_ts:
        partition_issues.append({"issue": "invalid_timestamp", "rows": invalid_ts})

    valid_ts = frame.dropna(subset=["timestamp"]).sort_values("timestamp")
    duplicate_ts = int(valid_ts["timestamp"].duplicated().sum())
    if duplicate_ts:
        partition_issues.append({"issue": "duplicate_timestamp", "rows": duplicate_ts})

    day_start = pd.Timestamp(date, tz="UTC")
    day_end = day_start + pd.Timedelta(days=1)
    outside_day = int((~valid_ts["timestamp"].between(day_start, day_end, inclusive="left")).sum())
    if outside_day:
        partition_issues.append({"issue": "timestamp_outside_partition_day", "rows": outside_day})

    missing_runs = compact_timestamp_runs(valid_ts["timestamp"])
    for run in missing_runs[:20]:
        partition_issues.append({"issue": "timestamp_gap_inside_partition", **run})

    if expected_rows is not None and len(frame) != expected_rows:
        partition_issues.append(
            {"issue": "unexpected_row_count", "expected_rows": expected_rows, "actual_rows": len(frame)}
        )

    numeric = frame.select_dtypes(include=[np.number])
    for column in numeric.columns:
        values = numeric[column].to_numpy(dtype="float64", copy=False)
        bad = np.isinf(values)
        if bad.any():
            finite_issues.append(
                {
                    "dataset": dataset_name,
                    "date": date,
                    "column": column,
                    "inf_rows": int(bad.sum()),
                }
            )

    summary = {
        "dataset": dataset_name,
        "date": date,
        "rows": len(frame),
        "columns": len(frame.columns),
        "timestamp_min": None if valid_ts.empty else valid_ts["timestamp"].min().isoformat(),
        "timestamp_max": None if valid_ts.empty else valid_ts["timestamp"].max().isoformat(),
        "issues": len(partition_issues),
    }
    return summary, partition_issues, finite_issues

In [8]:
def audit_dataset(dataset_name: str, config: dict) -> dict[str, pd.DataFrame]:
    print(f"\n=== {dataset_name} ===")
    inventory = list_dataset_partitions(BUCKET, config)
    if inventory.empty:
        raise FileNotFoundError(f"No parquet partitions found at s3://{BUCKET}/{dataset_prefix(config)}")

    dates_expected = expected_dates(inventory["date"].tolist())
    found_dates = set(inventory["date"])
    missing_dates = [day.strftime("%Y-%m-%d") for day in dates_expected if day.strftime("%Y-%m-%d") not in found_dates]
    missing_partitions = pd.DataFrame(
        [{"dataset": dataset_name, "date": date, "issue": "missing_partition"} for date in missing_dates]
    )

    partition_rows = []
    partition_issue_rows = []
    finite_issue_rows = []
    null_issue_rows = []
    timestamp_continuity_rows = []
    column_states: dict[str, ColumnState] = {}
    previous_ts: pd.Timestamp | None = None

    selected_inventory = inventory[inventory["date"].isin([d.strftime("%Y-%m-%d") for d in dates_expected])]
    total = len(selected_inventory)
    for pos, row in enumerate(selected_inventory.itertuples(index=False), start=1):
        date = row.date
        key = row.key
        if pos == 1 or pos % 50 == 0 or pos == total:
            print(f"{dataset_name}: {pos}/{total} {date}")

        frame = read_s3_parquet(BUCKET, key)
        summary, partition_issues, finite_issues = check_partition_frame(
            frame,
            dataset_name,
            date,
            config.get("expected_rows_per_full_day"),
        )
        partition_rows.append({**summary, "key": key, "size_bytes": row.size_bytes})
        partition_issue_rows.extend({"dataset": dataset_name, "date": date, **issue} for issue in partition_issues)
        finite_issue_rows.extend(finite_issues)

        frame = frame.copy()
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True, errors="coerce")
        frame = frame.dropna(subset=["timestamp"]).sort_values("timestamp").drop_duplicates("timestamp")

        if not frame.empty:
            current_min = frame["timestamp"].iloc[0]
            current_max = frame["timestamp"].iloc[-1]
            if previous_ts is not None and current_min != previous_ts + pd.Timedelta(EXPECTED_FREQ):
                expected_next = previous_ts + pd.Timedelta(EXPECTED_FREQ)
                gap_end = current_min - pd.Timedelta(EXPECTED_FREQ)
                missing = max(0, int((current_min - expected_next) / pd.Timedelta(EXPECTED_FREQ)))
                timestamp_continuity_rows.append(
                    {
                        "dataset": dataset_name,
                        "previous_timestamp": previous_ts.isoformat(),
                        "next_timestamp": current_min.isoformat(),
                        "expected_next_timestamp": expected_next.isoformat(),
                        "gap_end": gap_end.isoformat(),
                        "missing_rows": missing,
                    }
                )
            previous_ts = current_max

        null_issue_rows.extend(update_null_state(column_states, frame, dataset_name, date))

    null_summary = summarize_null_states(dataset_name, column_states)
    return {
        "inventory": inventory,
        "missing_partitions": missing_partitions,
        "partition_report": pd.DataFrame(partition_rows),
        "partition_issues": pd.DataFrame(partition_issue_rows),
        "timestamp_continuity_issues": pd.DataFrame(timestamp_continuity_rows),
        "null_gap_partition_issues": pd.DataFrame(null_issue_rows[:MAX_EXAMPLES_PER_DATASET]),
        "null_gap_summary": null_summary,
        "finite_issues": pd.DataFrame(finite_issue_rows),
    }

In [9]:
reports = {name: audit_dataset(name, config) for name, config in DATASETS.items()}


=== unified_dataset ===
unified_dataset: 1/2193 2020-02-01
unified_dataset: 50/2193 2020-03-21
unified_dataset: 100/2193 2020-05-10
unified_dataset: 150/2193 2020-06-29
unified_dataset: 200/2193 2020-08-18
unified_dataset: 250/2193 2020-10-07
unified_dataset: 300/2193 2020-11-26
unified_dataset: 350/2193 2021-01-15
unified_dataset: 400/2193 2021-03-06
unified_dataset: 450/2193 2021-04-25
unified_dataset: 500/2193 2021-06-14
unified_dataset: 550/2193 2021-08-03
unified_dataset: 600/2193 2021-09-22
unified_dataset: 650/2193 2021-11-11
unified_dataset: 700/2193 2021-12-31
unified_dataset: 750/2193 2022-02-19
unified_dataset: 800/2193 2022-04-10
unified_dataset: 850/2193 2022-05-30
unified_dataset: 900/2193 2022-07-19
unified_dataset: 950/2193 2022-09-07
unified_dataset: 1000/2193 2022-10-27
unified_dataset: 1050/2193 2022-12-16
unified_dataset: 1100/2193 2023-02-04
unified_dataset: 1150/2193 2023-03-26
unified_dataset: 1200/2193 2023-05-15
unified_dataset: 1250/2193 2023-07-04
unified_da

In [10]:
def concat_report(report_name: str) -> pd.DataFrame:
    frames = [dataset_reports[report_name] for dataset_reports in reports.values() if not dataset_reports[report_name].empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


combined = {
    "missing_partitions": concat_report("missing_partitions"),
    "partition_report": concat_report("partition_report"),
    "partition_issues": concat_report("partition_issues"),
    "timestamp_continuity_issues": concat_report("timestamp_continuity_issues"),
    "null_gap_partition_issues": concat_report("null_gap_partition_issues"),
    "null_gap_summary": concat_report("null_gap_summary"),
    "finite_issues": concat_report("finite_issues"),
}

summary_rows = []
for dataset_name, dataset_reports in reports.items():
    null_summary = dataset_reports["null_gap_summary"]
    failing_columns = 0 if null_summary.empty else int((null_summary["status"] != "ok").sum())
    summary_rows.append(
        {
            "dataset": dataset_name,
            "partitions": len(dataset_reports["inventory"]),
            "missing_partitions": len(dataset_reports["missing_partitions"]),
            "partition_issues": len(dataset_reports["partition_issues"]),
            "timestamp_continuity_issues": len(dataset_reports["timestamp_continuity_issues"]),
            "columns_with_null_gaps_or_all_null": failing_columns,
            "finite_issues": len(dataset_reports["finite_issues"]),
        }
    )

audit_summary = pd.DataFrame(summary_rows)
display(audit_summary)

,dataset,partitions,missing_partitions,partition_issues,timestamp_continuity_issues,columns_with_null_gaps_or_all_null,finite_issues
0,unified_dataset,2193,0,0,0,0,0
1,hmm_dataset,2193,0,1,0,0,0


In [11]:
for name, frame in combined.items():
    print(f"\n{name}: {len(frame)}")
    if not frame.empty:
        display(frame.head(50))


missing_partitions: 0

partition_report: 4386


,dataset,date,rows,columns,timestamp_min,timestamp_max,issues,key,size_bytes
0,unified_dataset,2020-02-01,1440,126,2020-02-01T00:00:00+00:00,2020-02-01T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1295663
1,unified_dataset,2020-02-02,1440,126,2020-02-02T00:00:00+00:00,2020-02-02T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1293384
2,unified_dataset,2020-02-03,1440,126,2020-02-03T00:00:00+00:00,2020-02-03T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1279588
3,unified_dataset,2020-02-04,1440,126,2020-02-04T00:00:00+00:00,2020-02-04T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1241410
4,unified_dataset,2020-02-05,1440,126,2020-02-05T00:00:00+00:00,2020-02-05T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1276147
5,unified_dataset,2020-02-06,1440,126,2020-02-06T00:00:00+00:00,2020-02-06T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1269764
6,unified_dataset,2020-02-07,1440,126,2020-02-07T00:00:00+00:00,2020-02-07T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1259396
7,unified_dataset,2020-02-08,1440,126,2020-02-08T00:00:00+00:00,2020-02-08T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1257638
8,unified_dataset,2020-02-09,1440,126,2020-02-09T00:00:00+00:00,2020-02-09T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1223521
9,unified_dataset,2020-02-10,1440,126,2020-02-10T00:00:00+00:00,2020-02-10T23:59:00+00:00,0,features/unified_dataset/symbol=ADAUSDT/interv...,1263204



partition_issues: 1


,dataset,date,issue,expected_rows,actual_rows
0,hmm_dataset,2020-02-01,unexpected_row_count,1440,1260



timestamp_continuity_issues: 0

null_gap_partition_issues: 0

null_gap_summary: 180


,dataset,column,status,total_rows,non_null_rows,leading_null_rows_allowed,gap_rows_after_first_valid,first_valid_ts,first_gap_ts,gap_examples
0,unified_dataset,ada_aggression_delta_norm_10m,ok,3157920,3157911,9,0,2020-02-01T00:09:00+00:00,None,[]
1,unified_dataset,ada_aggression_delta_norm_20m,ok,3157920,3157901,19,0,2020-02-01T00:19:00+00:00,None,[]
2,unified_dataset,ada_aggression_delta_norm_30m,ok,3157920,3157891,29,0,2020-02-01T00:29:00+00:00,None,[]
3,unified_dataset,ada_aggression_delta_norm_5m,ok,3157920,3157916,4,0,2020-02-01T00:04:00+00:00,None,[]
4,unified_dataset,ada_average_trade_size_quote_10m,ok,3157920,3157911,9,0,2020-02-01T00:09:00+00:00,None,[]
5,unified_dataset,ada_average_trade_size_quote_20m,ok,3157920,3157901,19,0,2020-02-01T00:19:00+00:00,None,[]
6,unified_dataset,ada_average_trade_size_quote_30m,ok,3157920,3157891,29,0,2020-02-01T00:29:00+00:00,None,[]
7,unified_dataset,ada_average_trade_size_quote_5m,ok,3157920,3157916,4,0,2020-02-01T00:04:00+00:00,None,[]
8,unified_dataset,ada_close_position_10m,ok,3157920,3157911,9,0,2020-02-01T00:09:00+00:00,None,[]
9,unified_dataset,ada_close_position_20m,ok,3157920,3157901,19,0,2020-02-01T00:19:00+00:00,None,[]



finite_issues: 0


In [12]:
for name, frame in {"audit_summary": audit_summary, **combined}.items():
    path = REPORT_DIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    print(path.resolve())

failed = audit_summary.drop(columns=["partitions"]).sum(numeric_only=True).sum() > 0
if failed:
    raise AssertionError("S3 dataset integrity audit found issues. See tables above and CSV reports.")

print("OK: integrity audit passed for all configured datasets.")

C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\audit_summary.csv
C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\missing_partitions.csv
C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\partition_report.csv
C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\partition_issues.csv
C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\timestamp_continuity_issues.csv
C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\null_gap_partition_issues.csv
C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\null_gap_summary.csv
C:\projects\binance-dowloader-3.0\analysis\data_integrity_reports\finite_issues.csv


AssertionError: S3 dataset integrity audit found issues. See tables above and CSV reports.

In [13]:
import io
import gc
import numpy as np
import pandas as pd

COMPACT_OUTPUTS = {
    "unified_dataset": {
        "local_path": "unified_dataset_full.parquet",
        "s3_key": "features/compact/unified_dataset_full.parquet",
    },
    "hmm_dataset": {
        "local_path": "hmm_dataset_full.parquet",
        "s3_key": "features/compact/hmm_dataset_full.parquet",
    },
}


def load_full_dataset(dataset_name: str, config: dict) -> pd.DataFrame:
    inventory = list_dataset_partitions(BUCKET, config)
    if inventory.empty:
        raise FileNotFoundError(f"{dataset_name}: no partitions found")

    frames = []
    total = len(inventory)

    for pos, row in enumerate(inventory.itertuples(index=False), start=1):
        if pos == 1 or pos % 100 == 0 or pos == total:
            print(f"{dataset_name}: loading {pos}/{total} {row.date}")

        part = read_s3_parquet(BUCKET, row.key)
        part["timestamp"] = pd.to_datetime(part["timestamp"], utc=True, errors="raise")
        frames.append(part)

    full = pd.concat(frames, ignore_index=True)
    full = full.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    del frames
    gc.collect()

    print(
        f"{dataset_name}: loaded rows={len(full):,}, "
        f"range={full['timestamp'].iloc[0]} -> {full['timestamp'].iloc[-1]}, "
        f"columns={len(full.columns)}"
    )

    return full


def first_complete_timestamp(frame: pd.DataFrame, dataset_name: str) -> pd.Timestamp:
    value_columns = [col for col in frame.columns if col != "timestamp"]
    complete_mask = frame[value_columns].notna().all(axis=1)

    if not complete_mask.any():
        raise ValueError(f"{dataset_name}: no fully complete rows found")

    return frame.loc[complete_mask, "timestamp"].iloc[0]


def assert_minute_continuity(frame: pd.DataFrame, dataset_name: str) -> None:
    ts = pd.DatetimeIndex(frame["timestamp"])

    if ts.has_duplicates:
        duplicated = ts[ts.duplicated()].unique()[:10]
        raise ValueError(f"{dataset_name}: duplicate timestamps: {list(duplicated)}")

    expected = pd.date_range(ts.min(), ts.max(), freq="1min", tz="UTC")
    if len(expected) != len(ts) or not ts.equals(expected):
        missing = expected.difference(ts)
        raise ValueError(
            f"{dataset_name}: timestamp is not continuous. "
            f"missing={len(missing)}, "
            f"first_missing={None if len(missing) == 0 else missing[0]}"
        )


def assert_no_missing_or_inf(frame: pd.DataFrame, dataset_name: str) -> None:
    missing_by_column = frame.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0]

    if not missing_by_column.empty:
        display(missing_by_column.sort_values(ascending=False).head(50))
        raise ValueError(f"{dataset_name}: output still contains missing values")

    numeric = frame.select_dtypes(include=[np.number])
    inf_by_column = {}

    for column in numeric.columns:
        values = numeric[column].to_numpy(dtype="float64", copy=False)
        count = int(np.isinf(values).sum())
        if count:
            inf_by_column[column] = count

    if inf_by_column:
        display(pd.Series(inf_by_column).sort_values(ascending=False).head(50))
        raise ValueError(f"{dataset_name}: output contains inf/-inf values")


def save_local_and_upload(dataset_name: str, frame: pd.DataFrame) -> None:
    output = COMPACT_OUTPUTS[dataset_name]
    local_path = output["local_path"]
    s3_key = output["s3_key"]

    frame.to_parquet(
        local_path,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    buffer = io.BytesIO()
    frame.to_parquet(
        buffer,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )
    payload = buffer.getvalue()

    s3.put_object(
        Bucket=BUCKET,
        Key=s3_key,
        Body=payload,
        ContentType="application/vnd.apache.parquet",
        Metadata={
            "dataset-kind": dataset_name,
            "rows": str(len(frame)),
            "columns": str(len(frame.columns)),
            "timestamp-start": frame["timestamp"].iloc[0].isoformat(),
            "timestamp-end": frame["timestamp"].iloc[-1].isoformat(),
            "frequency": "1m",
            "missing-values": "0",
            "aligned-with": "unified_dataset,hmm_dataset",
        },
    )

    print(
        f"{dataset_name}: saved {local_path}, "
        f"uploaded s3://{BUCKET}/{s3_key}, "
        f"rows={len(frame):,}, "
        f"size_mib={len(payload) / 1024**2:.2f}"
    )


raw_frames = {}

for dataset_name in ("unified_dataset", "hmm_dataset"):
    raw_frames[dataset_name] = load_full_dataset(dataset_name, DATASETS[dataset_name])

starts = {
    dataset_name: first_complete_timestamp(frame, dataset_name)
    for dataset_name, frame in raw_frames.items()
}

common_start = max(starts.values())

print("First complete timestamps:")
for dataset_name, start in starts.items():
    print(f"{dataset_name}: {start}")

print(f"Common trim start: {common_start}")

compact_frames = {}

for dataset_name, frame in raw_frames.items():
    compact = frame.loc[frame["timestamp"] >= common_start].reset_index(drop=True)

    assert_minute_continuity(compact, dataset_name)
    assert_no_missing_or_inf(compact, dataset_name)

    compact_frames[dataset_name] = compact

unified_ts = compact_frames["unified_dataset"]["timestamp"].reset_index(drop=True)
hmm_ts = compact_frames["hmm_dataset"]["timestamp"].reset_index(drop=True)

if not unified_ts.equals(hmm_ts):
    mismatch = unified_ts.ne(hmm_ts)
    first_mismatch = mismatch.idxmax() if mismatch.any() else None

    raise ValueError(
        "Timestamp mismatch after common trim. "
        f"unified_rows={len(unified_ts):,}, "
        f"hmm_rows={len(hmm_ts):,}, "
        f"first_mismatch={first_mismatch}, "
        f"unified_range=({unified_ts.iloc[0]}, {unified_ts.iloc[-1]}), "
        f"hmm_range=({hmm_ts.iloc[0]}, {hmm_ts.iloc[-1]})"
    )

print(
    "Aligned timestamp index OK: "
    f"rows={len(unified_ts):,}, "
    f"range={unified_ts.iloc[0]} -> {unified_ts.iloc[-1]}"
)

for dataset_name, frame in compact_frames.items():
    save_local_and_upload(dataset_name, frame)

del raw_frames
gc.collect()

print("Done.")

unified_dataset: loading 1/2193 2020-02-01
unified_dataset: loading 100/2193 2020-05-10
unified_dataset: loading 200/2193 2020-08-18
unified_dataset: loading 300/2193 2020-11-26
unified_dataset: loading 400/2193 2021-03-06
unified_dataset: loading 500/2193 2021-06-14
unified_dataset: loading 600/2193 2021-09-22
unified_dataset: loading 700/2193 2021-12-31
unified_dataset: loading 800/2193 2022-04-10
unified_dataset: loading 900/2193 2022-07-19
unified_dataset: loading 1000/2193 2022-10-27
unified_dataset: loading 1100/2193 2023-02-04
unified_dataset: loading 1200/2193 2023-05-15
unified_dataset: loading 1300/2193 2023-08-23
unified_dataset: loading 1400/2193 2023-12-01
unified_dataset: loading 1500/2193 2024-03-10
unified_dataset: loading 1600/2193 2024-06-18
unified_dataset: loading 1700/2193 2024-09-26
unified_dataset: loading 1800/2193 2025-01-04
unified_dataset: loading 1900/2193 2025-04-14
unified_dataset: loading 2000/2193 2025-07-23
unified_dataset: loading 2100/2193 2025-10-31


ReadTimeoutError: Read timeout on endpoint URL: "https://storage.yandexcloud.net/binance-data-downloader/features/compact/unified_dataset_full.parquet"

In [14]:
import os
import boto3
import pandas as pd
import pyarrow.parquet as pq

from botocore.config import Config
from boto3.s3.transfer import TransferConfig
from dotenv import load_dotenv

load_dotenv(".env")

BUCKET = os.getenv("YC_BUCKET", "binance-data-downloader")

upload_s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    config=Config(
        connect_timeout=60,
        read_timeout=600,
        retries={"max_attempts": 10, "mode": "standard"},
    ),
)

transfer_config = TransferConfig(
    multipart_threshold=64 * 1024 * 1024,
    multipart_chunksize=64 * 1024 * 1024,
    max_concurrency=4,
    use_threads=True,
)

UPLOADS = {
    "unified_dataset": {
        "local_path": "unified_dataset_full.parquet",
        "s3_key": "features/compact/unified_dataset_full.parquet",
    },
    "hmm_dataset": {
        "local_path": "hmm_dataset_full.parquet",
        "s3_key": "features/compact/hmm_dataset_full.parquet",
    },
}


def parquet_file_info(path: str) -> dict:
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    timestamp_col = pq.read_table(path, columns=["timestamp"]).to_pandas()["timestamp"]
    timestamp_col = pd.to_datetime(timestamp_col, utc=True)

    return {
        "rows": parquet.metadata.num_rows,
        "columns": len(schema.names),
        "timestamp_start": timestamp_col.iloc[0].isoformat(),
        "timestamp_end": timestamp_col.iloc[-1].isoformat(),
    }


for dataset_name, item in UPLOADS.items():
    local_path = item["local_path"]
    s3_key = item["s3_key"]

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local file not found: {local_path}")

    info = parquet_file_info(local_path)
    size_mib = os.path.getsize(local_path) / 1024**2

    print(
        f"{dataset_name}: uploading {local_path} -> s3://{BUCKET}/{s3_key}, "
        f"size_mib={size_mib:.2f}, rows={info['rows']:,}"
    )

    upload_s3.upload_file(
        Filename=local_path,
        Bucket=BUCKET,
        Key=s3_key,
        ExtraArgs={
            "ContentType": "application/vnd.apache.parquet",
            "Metadata": {
                "dataset-kind": dataset_name,
                "rows": str(info["rows"]),
                "columns": str(info["columns"]),
                "timestamp-start": info["timestamp_start"],
                "timestamp-end": info["timestamp_end"],
                "frequency": "1m",
                "missing-values": "0",
                "aligned-with": "unified_dataset,hmm_dataset",
            },
        },
        Config=transfer_config,
    )

    print(f"{dataset_name}: uploaded OK")

print("Done.")

unified_dataset: uploading unified_dataset_full.parquet -> s3://binance-data-downloader/features/compact/unified_dataset_full.parquet, size_mib=2035.07, rows=3,157,740
unified_dataset: uploaded OK


FileNotFoundError: Local file not found: hmm_dataset_full.parquet

In [15]:
import os
import gc
import boto3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from botocore.config import Config
from boto3.s3.transfer import TransferConfig
from dotenv import load_dotenv

if "compact_frames" not in globals():
    raise RuntimeError(
        "compact_frames not found in memory. "
        "Если kernel был перезапущен, придется заново собрать compact_frames."
    )

required = {"unified_dataset", "hmm_dataset"}
missing_in_memory = required.difference(compact_frames.keys())
if missing_in_memory:
    raise RuntimeError(f"compact_frames missing datasets: {sorted(missing_in_memory)}")


UPLOADS = {
    "unified_dataset": {
        "local_path": "unified_dataset_full.parquet",
        "s3_key": "features/compact/unified_dataset_full.parquet",
    },
    "hmm_dataset": {
        "local_path": "hmm_dataset_full.parquet",
        "s3_key": "features/compact/hmm_dataset_full.parquet",
    },
}


def assert_no_missing_or_inf_quick(frame: pd.DataFrame, dataset_name: str) -> None:
    missing = frame.isna().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        display(missing.sort_values(ascending=False).head(50))
        raise ValueError(f"{dataset_name}: contains missing values")

    numeric = frame.select_dtypes(include=[np.number])
    inf_counts = {}
    for column in numeric.columns:
        values = numeric[column].to_numpy(dtype="float64", copy=False)
        count = int(np.isinf(values).sum())
        if count:
            inf_counts[column] = count

    if inf_counts:
        display(pd.Series(inf_counts).sort_values(ascending=False).head(50))
        raise ValueError(f"{dataset_name}: contains inf/-inf values")


def assert_same_timestamps(left: pd.DataFrame, right: pd.DataFrame) -> None:
    left_ts = left["timestamp"].reset_index(drop=True)
    right_ts = right["timestamp"].reset_index(drop=True)

    if not left_ts.equals(right_ts):
        raise ValueError(
            "Timestamp mismatch: "
            f"unified_rows={len(left_ts):,}, hmm_rows={len(right_ts):,}, "
            f"unified_range=({left_ts.iloc[0]}, {left_ts.iloc[-1]}), "
            f"hmm_range=({right_ts.iloc[0]}, {right_ts.iloc[-1]})"
        )


assert_same_timestamps(
    compact_frames["unified_dataset"],
    compact_frames["hmm_dataset"],
)

for dataset_name, frame in compact_frames.items():
    assert_no_missing_or_inf_quick(frame, dataset_name)

for dataset_name, item in UPLOADS.items():
    local_path = item["local_path"]

    if os.path.exists(local_path):
        print(f"{dataset_name}: local file already exists: {local_path}")
        continue

    print(f"{dataset_name}: saving missing local file: {local_path}")

    compact_frames[dataset_name].to_parquet(
        local_path,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    print(
        f"{dataset_name}: saved {local_path}, "
        f"size_mib={os.path.getsize(local_path) / 1024**2:.2f}"
    )

gc.collect()


load_dotenv(".env")

BUCKET = os.getenv("YC_BUCKET", "binance-data-downloader")

upload_s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    config=Config(
        connect_timeout=60,
        read_timeout=600,
        retries={"max_attempts": 10, "mode": "standard"},
    ),
)

transfer_config = TransferConfig(
    multipart_threshold=64 * 1024 * 1024,
    multipart_chunksize=64 * 1024 * 1024,
    max_concurrency=4,
    use_threads=True,
)


def parquet_file_info(path: str) -> dict:
    parquet = pq.ParquetFile(path)
    timestamp_col = pq.read_table(path, columns=["timestamp"]).to_pandas()["timestamp"]
    timestamp_col = pd.to_datetime(timestamp_col, utc=True)

    return {
        "rows": parquet.metadata.num_rows,
        "columns": len(parquet.schema_arrow.names),
        "timestamp_start": timestamp_col.iloc[0].isoformat(),
        "timestamp_end": timestamp_col.iloc[-1].isoformat(),
    }


for dataset_name, item in UPLOADS.items():
    local_path = item["local_path"]
    s3_key = item["s3_key"]

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local file not found after save attempt: {local_path}")

    info = parquet_file_info(local_path)
    size_mib = os.path.getsize(local_path) / 1024**2

    print(
        f"{dataset_name}: uploading {local_path} -> s3://{BUCKET}/{s3_key}, "
        f"size_mib={size_mib:.2f}, rows={info['rows']:,}"
    )

    upload_s3.upload_file(
        Filename=local_path,
        Bucket=BUCKET,
        Key=s3_key,
        ExtraArgs={
            "ContentType": "application/vnd.apache.parquet",
            "Metadata": {
                "dataset-kind": dataset_name,
                "rows": str(info["rows"]),
                "columns": str(info["columns"]),
                "timestamp-start": info["timestamp_start"],
                "timestamp-end": info["timestamp_end"],
                "frequency": "1m",
                "missing-values": "0",
                "aligned-with": "unified_dataset,hmm_dataset",
            },
        },
        Config=transfer_config,
    )

    print(f"{dataset_name}: uploaded OK")

print("Done.")

unified_dataset: local file already exists: unified_dataset_full.parquet
hmm_dataset: saving missing local file: hmm_dataset_full.parquet
hmm_dataset: saved hmm_dataset_full.parquet, size_mib=663.57
unified_dataset: uploading unified_dataset_full.parquet -> s3://binance-data-downloader/features/compact/unified_dataset_full.parquet, size_mib=2035.07, rows=3,157,740
unified_dataset: uploaded OK
hmm_dataset: uploading hmm_dataset_full.parquet -> s3://binance-data-downloader/features/compact/hmm_dataset_full.parquet, size_mib=663.57, rows=3,157,740
hmm_dataset: uploaded OK
Done.
